In [14]:
import os
import re
import csv

INPUT_FOLDER = r"C:\Users\Aqil Anhein\Desktop\TA\Peraturan Pemerintah\cleaned_txt\final_cleaned"
OUTPUT_CSV = r"C:\Users\Aqil Anhein\Desktop\TA\Peraturan Pemerintah\cleaned_txt\final_cleaned\extracted_legal_data.csv"

def extract_bab(text):
    """
    Extract all BABs with format like 'BAB X : TITLE' or 'BAB II - TITLE'
    Returns a list of tuples (bab_number, bab_title, start_index)
    """
    bab_pattern = r'\bBAB\s+([IVXLCDM\d]+)[\s\-:]+(.+?)(?=\n|$)'
    return [(m.group(1), m.group(2).strip(), m.start()) for m in re.finditer(bab_pattern, text, re.IGNORECASE)]

def extract_pasal_blocks(text):
    """
    Extract PASAL sections and their content
    Returns a list of tuples (pasal_number, content)
    """
    pasal_pattern = r'(PASAL\s+\d+)\s*\n?'
    matches = list(re.finditer(pasal_pattern, text))
    pasal_blocks = []

    for i, match in enumerate(matches):
        pasal_title = match.group(1).strip()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        content = text[start:end].strip()
        pasal_blocks.append((pasal_title, content))

    return pasal_blocks

def find_bab_for_pasal(pasal_index, bab_list):
    """Find the most recent BAB before the given pasal index"""
    current_bab = None
    for bab_num, bab_title, bab_index in bab_list:
        if bab_index < pasal_index:
            current_bab = f'BAB {bab_num}: {bab_title}'
        else:
            break
    return current_bab or "UNKNOWN"

def process_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    bab_list = extract_bab(text)
    pasal_blocks = extract_pasal_blocks(text)

    rows = []
    for pasal_title, content in pasal_blocks:
        pasal_index = text.find(pasal_title)
        bab_context = find_bab_for_pasal(pasal_index, bab_list)
        rows.append({
            "judul": os.path.basename(file_path),
            "BAB": bab_context,
            "Pasal": pasal_title,
            "Content": content.replace("\n", " ").strip()
        })

    return rows

# Process all files and write to CSV
all_rows = []
for filename in os.listdir(INPUT_FOLDER):
    if filename.endswith(".txt"):
        file_path = os.path.join(INPUT_FOLDER, filename)
        extracted = process_file(file_path)
        all_rows.extend(extracted)

# Write to CSV
with open(OUTPUT_CSV, "w", newline='', encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=["judul", "BAB", "Pasal", "Content"])
    writer.writeheader()
    writer.writerows(all_rows)

print(f"✅ Extraction complete. Output saved to: {OUTPUT_CSV}")


✅ Extraction complete. Output saved to: C:\Users\Aqil Anhein\Desktop\TA\Peraturan Pemerintah\cleaned_txt\final_cleaned\extracted_legal_data.csv
